# Discover and download a Data Registry document

This notebook accompanies Step 3 of the RHOAI 3.6 Data Registry walkthrough. It uses the `data-registry-example` project and the fictional FAQ registered in Step 2.

**What you will prove:** an authorized workbench can read Data Registry metadata over verified TLS, see the asset's S3 connection reference, use its *attached* connection to download the object, and verify its SHA-256. You will also register a second example collection and asset through the API, then read them back. The API writes are safe to rerun only when the existing resources match this notebook's marker and metadata.

Before running, mount the project connection Secret `data-registry-storage` in this workspace, grant its actual ServiceAccount the `feast-data-registry-editor` role in this project, and confirm its projected public `service-ca.crt` exists. The notebook never prints or saves a token or S3 key. Do not put them in a notebook cell or `.env` file.


In [ ]:
# Imports and optional public configuration from a local .env file.
import hashlib
import os
from pathlib import Path
from urllib.parse import urlparse

import boto3
from botocore.config import Config
from botocore.exceptions import BotoCoreError, ClientError
import requests
from dotenv import load_dotenv

load_dotenv(override=False)
print("Python dependencies are available.")


## 1. Configure the example and check the runtime

These are public names and paths, not credentials. The Service URL uses the DNS name on its OpenShift serving certificate. The projected `service-ca.crt` is the public OpenShift service CA mounted in the workspace pod. The expected hash is for the committed synthetic FAQ. Change the names only when you also change the Step 2 asset and uploaded object.


In [ ]:
PROJECT = os.getenv("REGISTRY_PROJECT", "data-registry-example")
COLLECTION = os.getenv("REGISTRY_COLLECTION", "customer-support")
ASSET_NAME = os.getenv("REGISTRY_ASSET", "customer-support-faq")
CONNECTION_NAME = os.getenv("RHOAI_CONNECTION_NAME", "data-registry-storage")
REGISTRY_URL = os.getenv(
    "REGISTRY_URL",
    "https://feast-data-registry-data-registry.rhoai-data-registry.svc:443",
).rstrip("/")
CA_BUNDLE = Path(os.getenv("REGISTRY_CA_BUNDLE", "/var/run/secrets/kubernetes.io/serviceaccount/service-ca.crt"))
CONNECTION_MOUNT = Path(os.getenv("RHOAI_CONNECTION_MOUNT", "/var/run/secrets/data-registry-storage"))
TOKEN_FILE = Path("/var/run/secrets/kubernetes.io/serviceaccount/token")
S3_PREFIX = "examples/customer-support/"
EXPECTED_KEY = S3_PREFIX + "customer-support-faq.md"
EXPECTED_SHA256 = "e5be400a6740cb71a0ae93606a194f6cfcde5db09460b8c88c6e50afb0e57f78"

assert urlparse(REGISTRY_URL).scheme == "https", "Use the registry HTTPS Service."
assert CA_BUNDLE.is_file(), "The projected service-ca.crt is missing; use the public CA export described in Step 3."
assert TOKEN_FILE.is_file(), "The workbench must mount a projected ServiceAccount token."
def connection_value(name):
    """Read a project connection from its WorkspacesV2 Secret mount or legacy env injection."""
    secret_file = CONNECTION_MOUNT / name
    if secret_file.is_file():
        return secret_file.read_text(encoding="utf-8").strip()
    return os.getenv(name)

required_s3 = ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_S3_ENDPOINT", "AWS_DEFAULT_REGION", "AWS_S3_BUCKET"]
S3_CONFIG = {name: connection_value(name) for name in required_s3}
missing = [name for name, value in S3_CONFIG.items() if not value]
assert not missing, f"Mount the S3 connection Secret and restart the workspace; missing settings: {missing}"
print("Project:", PROJECT)
print("Registry Service:", REGISTRY_URL)
print("Public CA file:", CA_BUNDLE)
print("S3 connection settings present:", ", ".join(required_s3))


## 2. Create a small API helper

The projected ServiceAccount token belongs to this workbench. It is read into memory and sent only in the HTTPS `Authorization` header to the registry Service. `requests` validates the certificate against the public service CA. Error messages deliberately show the path and status, not a response body or authentication header.


In [ ]:
token = TOKEN_FILE.read_text(encoding="utf-8").strip()
assert token, "The projected ServiceAccount token is empty."
http = requests.Session()
http.trust_env = False  # In-cluster Service calls must not use a workstation proxy.
http.headers.update({"Authorization": f"Bearer {token}", "Accept": "application/json"})

def api_request(method, path, *, body=None, authenticated=True, expected=(200,)):
    session = http if authenticated else requests.Session()
    session.trust_env = False
    try:
        response = session.request(
            method,
            REGISTRY_URL + path,
            json=body,
            verify=str(CA_BUNDLE),
            timeout=20,
        )
    finally:
        if not authenticated:
            session.close()
    if response.status_code not in expected:
        raise RuntimeError(f"{method} {path}: HTTP {response.status_code}; expected {expected}. Check the Step 3 prerequisites.")
    if response.status_code in (204, 401, 403, 409):
        return response.status_code
    return response.json()

print("Authenticated, certificate-verified API client is ready.")


## 3. Check authentication and list collections

The TLS proxy denies `/v1/config` without a token. With the workspace token, the project-scoped `/v1/{project}/config` endpoint reports supported operations, and the collection endpoint lists `customer-support`. This separates service reachability from authorization.


In [ ]:
unauthenticated_status = api_request("GET", "/v1/config", authenticated=False, expected=(401, 403))
print("Unauthenticated discovery request status:", unauthenticated_status)
config = api_request("GET", f"/v1/{PROJECT}/config")
print("Registry discovery keys:", sorted(config))
collections_path = f"/v1/{PROJECT}/namespaces"
collections = api_request("GET", collections_path)
collection_names = [parts[0] for parts in collections["namespaces"]]
assert COLLECTION in collection_names, f"Collection {COLLECTION!r} was not found."
print("Collections:", collection_names)


## 4. Read the collection and asset metadata

The collection is an organizational group within the project. The asset is a *volume* because it describes unstructured documents. List volumes, select the exact asset name, then GET its detail endpoint. Do not use the first list row or guess a UUID: a project can contain many assets.


In [ ]:
collection_path = f"{collections_path}/{COLLECTION}"
collection = api_request("GET", collection_path)
assert collection["namespace"] == [COLLECTION]
print("Collection:", collection["namespace"][0])
print("Collection description:", collection.get("properties", {}).get("description", ""))
volumes_path = f"{collection_path}/volumes"
volumes = api_request("GET", volumes_path)["volumes"]
matching = [item for item in volumes if item.get("name") == ASSET_NAME]
assert len(matching) == 1, f"Expected exactly one {ASSET_NAME!r} volume; found {len(matching)}."
asset = api_request("GET", f"{volumes_path}/{ASSET_NAME}")
assert asset["name"] == ASSET_NAME
assert asset["format"] == "documents"
assert asset["storage_location"] == f"s3://{S3_CONFIG['AWS_S3_BUCKET']}/{S3_PREFIX}"
assert asset["connection_ref"]["type"] == "secret"
assert asset["connection_ref"]["secret_name"] == CONNECTION_NAME
print("Asset:", asset["name"])
print("Format:", asset["format"])
print("S3 path:", asset["storage_location"])
print("Connection reference:", asset["connection_ref"]["secret_name"])


## 5. Use the attached connection to list S3 objects

The registry response gives the bucket/prefix and a reference to `data-registry-storage`; it does **not** return the access key or secret key. WorkspacesV2 mounts the project connection Secret as files under `/var/run/secrets/data-registry-storage`; older workbenches may inject the same settings as environment variables. The helper above accepts either form without printing values. The client below supplies them explicitly to boto3, preventing a silent fallback to another AWS identity. The internal SeaweedFS endpoint uses HTTP inside the cluster; use HTTPS and a trusted CA for external object stores.


In [ ]:
endpoint = S3_CONFIG["AWS_S3_ENDPOINT"].rstrip("/")
parsed = urlparse(endpoint)
assert parsed.scheme in ("http", "https") and parsed.hostname and not parsed.username and not parsed.password
assert not parsed.path and not parsed.query, "Use the S3 API origin, without a bucket path."
if parsed.scheme == "http":
    assert parsed.hostname.endswith(".svc") or parsed.hostname.endswith(".svc.cluster.local"), "HTTP is only for an internal cluster Service."
bucket = S3_CONFIG["AWS_S3_BUCKET"]
s3 = boto3.client(
    "s3",
    endpoint_url=endpoint,
    region_name=S3_CONFIG["AWS_DEFAULT_REGION"],
    aws_access_key_id=S3_CONFIG["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=S3_CONFIG["AWS_SECRET_ACCESS_KEY"],
    aws_session_token=connection_value("AWS_SESSION_TOKEN"),
    config=Config(s3={"addressing_style": "path"}, retries={"max_attempts": 3}),
)
try:
    keys = []
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=S3_PREFIX):
        keys.extend(obj["Key"] for obj in page.get("Contents", []))
except (BotoCoreError, ClientError) as exc:
    raise RuntimeError("S3 list failed. Check connection attachment, bucket access, and storage networking.") from None
assert EXPECTED_KEY in keys, f"Expected fixture {EXPECTED_KEY!r} was not found under the asset prefix."
print("Object keys under the asset prefix:", keys)


## 6. Download the fixture and verify its bytes

Download only the expected key, keep it in the workbench's `downloads/` folder, and compare its SHA-256 to the synthetic fixture committed with Step 2. The checksum protects this example from accidentally claiming success for a different object. It is not a general malware or document-content inspection.


In [ ]:
download_dir = Path("downloads")
download_dir.mkdir(exist_ok=True)
local_file = download_dir / "customer-support-faq.md"
try:
    object_body = s3.get_object(Bucket=bucket, Key=EXPECTED_KEY)["Body"].read()
except (BotoCoreError, ClientError):
    raise RuntimeError("S3 download failed. Check the key and connection permissions.") from None
actual_sha256 = hashlib.sha256(object_body).hexdigest()
assert actual_sha256 == EXPECTED_SHA256, "The downloaded object differs from the committed synthetic fixture."
local_file.write_bytes(object_body)
print("Saved:", local_file.resolve())
print("Bytes:", len(object_body))
print("SHA-256:", actual_sha256)


## 7. Register a second collection and asset through the API

So far you have only read the UI-created metadata. Now make an API-created collection, then register a second asset that points at the same S3 prefix and connection. This is **metadata creation**: no object is uploaded or copied. The fixed `example.source` marker lets a rerun recognize this notebook's own resources. A name collision with different metadata stops the notebook instead of overwriting somebody else's work.


In [ ]:
API_COLLECTION = "customer-support-api"
API_ASSET = "customer-support-faq-api"
MARKER = "data-registry-api-walkthrough"
api_collection_path = f"{collections_path}/{API_COLLECTION}"
new_collection = {
    "namespace": [API_COLLECTION],
    "properties": {
        "description": "API-created collection for the Data Registry workbench walkthrough.",
        "example.source": MARKER,
    },
}
result = api_request("POST", collections_path, body=new_collection, expected=(200, 409))
if result == 409:
    existing = api_request("GET", api_collection_path)
    assert existing.get("properties", {}).get("example.source") == MARKER, "Collection name belongs to a different owner; choose a new name."
    print("Reused this notebook's existing API collection.")
else:
    print("Created API collection:", API_COLLECTION)
confirmed_collection = api_request("GET", api_collection_path)
assert confirmed_collection["namespace"] == [API_COLLECTION]


## 8. Register the API asset and read it back

The request body uses `format: documents`, `storage_location`, and the typed RHOAI `connection_ref`. The collection appears in the URL, not in the body. The server assigns the owner from the authenticated ServiceAccount. Read the asset back to confirm what was persisted; a successful POST alone is not enough.


In [ ]:
api_volumes_path = f"{api_collection_path}/volumes"
new_asset = {
    "name": API_ASSET,
    "format": "documents",
    "description": "API-created catalog entry for the fictional customer support FAQ.",
    "storage_location": f"s3://{bucket}/{S3_PREFIX}",
    "connection_ref": {"type": "secret", "secret_name": CONNECTION_NAME},
    "purpose": "Demonstrate API registration and verified document retrieval.",
    "license": "internal-use",
    "maturity": "experimental",
    "domain": "customer-support",
    "pii": "none",
    "labels": ["customer-support", "synthetic", "api-example"],
    "properties": {"example.source": MARKER},
}
result = api_request("POST", api_volumes_path, body=new_asset, expected=(200, 409))
if result == 409:
    print("API asset exists; checking that it belongs to this notebook.")
else:
    print("Created API asset:", API_ASSET)
confirmed = api_request("GET", f"{api_volumes_path}/{API_ASSET}")
assert confirmed["name"] == API_ASSET
assert confirmed["format"] == "documents"
assert confirmed["storage_location"] == new_asset["storage_location"]
assert confirmed["connection_ref"]["type"] == "secret"
assert confirmed["connection_ref"]["secret_name"] == CONNECTION_NAME
assert confirmed.get("properties", {}).get("example.source") == MARKER, "API asset name belongs to a different resource."
print("Read-back verified:", confirmed["name"], "in", API_COLLECTION)


## 9. Summary and next step

You have verified three independent layers: project-scoped registry authorization, catalog metadata and its typed connection reference, and an S3 object read with the attached connection. You also created and read back metadata through the API. The example leaves the API-created assets and downloaded file in place so you can inspect them in **AI hub → Data** and JupyterLab. Nothing here deletes persistent data.


In [ ]:
print("Data Registry API walkthrough complete")
print("UI-created asset:", f"{COLLECTION}/{ASSET_NAME}")
print("API-created asset:", f"{API_COLLECTION}/{API_ASSET}")
print("Verified object:", f"s3://{bucket}/{EXPECTED_KEY}")
print("Downloaded file:", local_file.resolve())
print("SHA-256 verified:", actual_sha256 == EXPECTED_SHA256)
